In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

In [2]:
INPUT_PATH  = "/Users/kamilaya/Desktop/thesis/tshirts_final_final.csv"
REPORT_PATH = "preprocessing_report.txt"

In [3]:
report_lines = []
def log(msg):
    print(msg)
    report_lines.append(msg)

In [4]:
log("=" * 70)
log("STEP 1 — Load & Inspect")
log("=" * 70)
 
df = pd.read_csv(INPUT_PATH, parse_dates=["week_start"])
log(f"  Shape:         {df.shape}")
log(f"  Date range:    {df['week_start'].min().date()}  →  {df['week_start'].max().date()}")
log(f"  Unique articles: {df['article_id'].nunique():,}")
log(f"  Unique weeks:    {df['week_start'].nunique()}")
log(f"\n  Column types:\n{df.dtypes.to_string()}")
log(f"\n  Null counts:\n{df.isnull().sum().to_string()}")
log(f"\n  Sales volume distribution:\n{df['weekly_sales_volume'].describe().to_string()}")
log(f"\n  index_group breakdown:\n{df['index_group_name'].value_counts().to_string()}")

STEP 1 — Load & Inspect
  Shape:         (151985, 24)
  Date range:    2018-09-17  →  2020-09-21
  Unique articles: 7,873
  Unique weeks:    106

  Column types:
article_id                              int64
week_start                     datetime64[us]
weekly_sales_volume                     int64
avg_weekly_price                      float64
year_month                                str
product_type_name                         str
product_group_name                        str
graphical_appearance_name                 str
colour_group_name                         str
perceived_colour_value_name               str
index_group_name                          str
eurozone_hicp                         float64
eurozone_unemployment_rate            float64
eurozone_cci                          float64
week_of_year                            int64
month                                   int64
quarter                                 int64
year                                    int64
is_summer 

In [5]:
log("\n" + "=" * 70)
log("STEP 2 — Clean")
log("=" * 70)
 
original_len = len(df)


STEP 2 — Clean


In [6]:
df = df.drop_duplicates(subset=["article_id", "week_start"])
log(f"  After dedup:          {len(df):,} rows  (removed {original_len - len(df):,})")

  After dedup:          151,985 rows  (removed 0)


In [7]:
# Drop rows where macro data is missing (weeks outside Eurostat coverage)
macro_cols = ["eurozone_hicp", "eurozone_unemployment_rate", "eurozone_cci"]
before = len(df)
df = df.dropna(subset=macro_cols)
log(f"  After dropping rows with missing macro: {len(df):,} (removed {before - len(df):,})")

  After dropping rows with missing macro: 151,985 (removed 0)


In [8]:
# Remove articles missing core attribute columns
attr_cols = ["product_type_name", "graphical_appearance_name",
             "colour_group_name", "perceived_colour_value_name", "index_group_name"]
before = len(df)
df = df.dropna(subset=attr_cols)
log(f"  After dropping rows with missing attributes: {len(df):,} (removed {before - len(df):,})")

  After dropping rows with missing attributes: 151,985 (removed 0)


In [9]:
# Outlier check
q99 = df["weekly_sales_volume"].quantile(0.99)
outliers = df[df["weekly_sales_volume"] > q99]
log(f"\n  Sales volume 99th percentile: {q99:.0f} units/week")
log(f"  Rows above 99th pct: {len(outliers):,}  ({len(outliers)/len(df)*100:.2f}%)")
# NOTE: We cap rather than remove, to preserve time-series continuity
df["weekly_sales_volume"] = df["weekly_sales_volume"].clip(upper=q99)
log(f"  → Capped (not dropped) at {q99:.0f} to preserve time-series integrity")


  Sales volume 99th percentile: 164 units/week
  Rows above 99th pct: 1,516  (1.00%)
  → Capped (not dropped) at 164 to preserve time-series integrity


In [10]:
log("\n" + "=" * 70)
log("STEP 3 — Zero-Week Imputation")
log("=" * 70)
 
all_weeks      = df["week_start"].unique()
all_articles   = df["article_id"].unique()
log(f"  Building full grid: {len(all_articles):,} articles × {len(all_weeks):,} weeks = {len(all_articles)*len(all_weeks):,} rows")
 
full_grid = pd.MultiIndex.from_product(
    [all_articles, sorted(all_weeks)],
    names=["article_id", "week_start"]
).to_frame(index=False)


STEP 3 — Zero-Week Imputation
  Building full grid: 7,873 articles × 106 weeks = 834,538 rows


In [11]:
df_full = full_grid.merge(df, on=["article_id", "week_start"], how="left")
df_full["weekly_sales_volume"] = df_full["weekly_sales_volume"].fillna(0)

In [12]:
df_full["avg_weekly_price"] = (
    df_full.groupby("article_id")["avg_weekly_price"]
    .transform(lambda x: x.fillna(x.median()))
)

In [13]:
for col in attr_cols + ["product_group_name"]:
    df_full[col] = df_full.groupby("article_id")[col].transform(
        lambda x: x.ffill().bfill()
    )

In [14]:
df_full["year_month"] = df_full["week_start"].dt.to_period("M").astype(str)

In [15]:
macro = df[["year_month"] + macro_cols].drop_duplicates()
df_full = df_full.drop(columns=macro_cols, errors="ignore")
df_full = df_full.merge(macro, on="year_month", how="left")
 
log(f"  Full grid shape after imputation: {df_full.shape}")
zero_pct = (df_full["weekly_sales_volume"] == 0).mean() * 100
log(f"  Zero-sale weeks: {zero_pct:.1f}%  (intermittent demand — expected in fashion)")

  Full grid shape after imputation: (834538, 24)
  Zero-sale weeks: 81.8%  (intermittent demand — expected in fashion)


In [16]:
log("\n" + "=" * 70)
log("STEP 4 — Feature Engineering")
log("=" * 70)
 
df_full = df_full.sort_values(["article_id", "week_start"]).reset_index(drop=True)


STEP 4 — Feature Engineering


In [17]:
df_full["week_of_year"] = df_full["week_start"].dt.isocalendar().week.astype(int)
df_full["month"]        = df_full["week_start"].dt.month
df_full["quarter"]      = df_full["week_start"].dt.quarter
df_full["year"]         = df_full["week_start"].dt.year

In [18]:
df_full["is_spring_summer"] = df_full["month"].isin([3, 4, 5, 6, 7, 8]).astype(int)
df_full["is_sale_season"]   = df_full["month"].isin([1, 7]).astype(int)

In [19]:
# COVID-19 dummy variable (supervisor recommendation)
# 1 from the week containing March 2020 onwards (WHO pandemic declaration)
df_full["covid"] = (df_full["week_start"] >= "2020-03-01").astype(int)
n_covid = df_full["covid"].sum()
log(f"  covid dummy: {n_covid:,} rows flagged as COVID period "
    f"({n_covid/len(df_full)*100:.1f}% of full grid)")

  covid dummy: 236,190 rows flagged as COVID period (28.3% of full grid)


In [20]:
first_sale = df_full[df_full["weekly_sales_volume"] > 0].groupby("article_id")["week_start"].min()
first_sale.name = "first_sale_week"
df_full = df_full.merge(first_sale, on="article_id", how="left")
df_full["product_age_weeks"] = (
    (df_full["week_start"] - df_full["first_sale_week"]).dt.days // 7
).clip(lower=0)
df_full.drop(columns=["first_sale_week"], inplace=True)

In [21]:
grp = df_full.groupby("article_id")["weekly_sales_volume"]
df_full["sales_lag1"] = grp.shift(1)
df_full["sales_lag2"] = grp.shift(2)
df_full["sales_lag4"] = grp.shift(4)

In [22]:
df_full["sales_rolling4_mean"] = grp.transform(
    lambda x: x.shift(1).rolling(4, min_periods=1).mean()
)
df_full["sales_rolling4_std"] = grp.transform(
    lambda x: x.shift(1).rolling(4, min_periods=1).std().fillna(0)
)

In [23]:
# Real price
df_full["real_price"] = df_full["avg_weekly_price"] / (1 + df_full["eurozone_hicp"] / 100)

In [24]:
# Relative price
article_median_price = df_full.groupby("article_id")["avg_weekly_price"].transform("median")
df_full["price_vs_median"] = df_full["avg_weekly_price"] / article_median_price.replace(0, np.nan)

In [25]:
macro_monthly = (
    df_full[["year_month"] + macro_cols]
    .drop_duplicates()
    .sort_values("year_month")
    .reset_index(drop=True)
)
macro_monthly["hicp_lag1m"]        = macro_monthly["eurozone_hicp"].shift(1)
macro_monthly["unemployment_lag1m"] = macro_monthly["eurozone_unemployment_rate"].shift(1)
macro_monthly["cci_lag1m"]         = macro_monthly["eurozone_cci"].shift(1)
 
df_full = df_full.merge(
    macro_monthly[["year_month", "hicp_lag1m", "unemployment_lag1m", "cci_lag1m"]],
    on="year_month", how="left"
)
 
log(f"  Features created. Current columns ({len(df_full.columns)}):\n  {df_full.columns.tolist()}")

  Features created. Current columns (34):
  ['article_id', 'week_start', 'weekly_sales_volume', 'avg_weekly_price', 'year_month', 'product_type_name', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name', 'index_group_name', 'week_of_year', 'month', 'quarter', 'year', 'is_summer', 'product_age_weeks', 'sales_lag1', 'sales_lag2', 'sales_rolling4', 'real_price', 'eurozone_hicp', 'eurozone_unemployment_rate', 'eurozone_cci', 'is_spring_summer', 'is_sale_season', 'covid', 'sales_lag4', 'sales_rolling4_mean', 'sales_rolling4_std', 'price_vs_median', 'hicp_lag1m', 'unemployment_lag1m', 'cci_lag1m']


In [26]:
lag_cols = ["sales_lag1", "sales_lag2", "sales_lag4",
            "sales_rolling4_mean", "sales_rolling4_std",
            "hicp_lag1m", "unemployment_lag1m", "cci_lag1m",
            "price_vs_median"]
df_full[lag_cols] = df_full[lag_cols].fillna(0)

In [27]:
# Categorical encoding
log("\n" + "=" * 70)
log("STEP 5 — Categorical Encoding")
log("=" * 70)
 
cat_cols = [
    "graphical_appearance_name",   # Solid, Melange, Stripe…
    "colour_group_name",           # Black, White, Blue…
    "perceived_colour_value_name", # Dark, Light, Medium, Dusty Light
    "index_group_name",            # Ladieswear, Baby/Children, Divided
]


STEP 5 — Categorical Encoding


In [28]:
for col in cat_cols:
    log(f"  {col}: {df_full[col].nunique()} unique values → one-hot encoded")

  graphical_appearance_name: 27 unique values → one-hot encoded
  colour_group_name: 46 unique values → one-hot encoded
  perceived_colour_value_name: 8 unique values → one-hot encoded
  index_group_name: 5 unique values → one-hot encoded


In [29]:
df_full = pd.get_dummies(df_full, columns=cat_cols, drop_first=True, dtype=int)
log(f"  Shape after encoding: {df_full.shape}")

  Shape after encoding: (834538, 112)


In [30]:
log("\n" + "=" * 70)
log("STEP 6 — Log-Transform Target")
log("=" * 70)
 
df_full["log_sales_volume"] = np.log1p(df_full["weekly_sales_volume"])
log(f"  Original sales:     mean={df_full['weekly_sales_volume'].mean():.2f}, "
    f"std={df_full['weekly_sales_volume'].std():.2f}, "
    f"skew={df_full['weekly_sales_volume'].skew():.2f}")
log(f"  Log-transformed:    mean={df_full['log_sales_volume'].mean():.2f}, "
    f"std={df_full['log_sales_volume'].std():.2f}, "
    f"skew={df_full['log_sales_volume'].skew():.2f}")


STEP 6 — Log-Transform Target
  Original sales:     mean=2.47, std=12.36, skew=8.75
  Log-transformed:    mean=0.34, std=0.87, skew=2.98


In [31]:
log("\n" + "=" * 70)
log("STEP 7 — Time-Based Train / Val / Test Split")
log("=" * 70)
 
TRAIN_END = "2019-04-30"
VAL_END   = "2019-12-31"
 
train = df_full[df_full["week_start"] <= TRAIN_END].copy()
val   = df_full[(df_full["week_start"] > TRAIN_END) & (df_full["week_start"] <= VAL_END)].copy()
test  = df_full[df_full["week_start"] > VAL_END].copy()
 
log(f"  Train: {train['week_start'].min().date()} → {train['week_start'].max().date()}  |  {len(train):,} rows")
log(f"  Val:   {val['week_start'].min().date()}   → {val['week_start'].max().date()}    |  {len(val):,} rows")
log(f"  Test:  {test['week_start'].min().date()}  → {test['week_start'].max().date()}   |  {len(test):,} rows")
log(f"  Split ratios: {len(train)/len(df_full):.0%} / {len(val)/len(df_full):.0%} / {len(test)/len(df_full):.0%}")


STEP 7 — Time-Based Train / Val / Test Split
  Train: 2018-09-17 → 2019-04-29  |  259,809 rows
  Val:   2019-05-06   → 2019-12-30    |  275,555 rows
  Test:  2020-01-06  → 2020-09-21   |  299,174 rows
  Split ratios: 31% / 33% / 36%


In [32]:
log("\n" + "=" * 70)
log("STEP 8 — Standardise Continuous Features (fit on train only)")
log("=" * 70)
 
scale_cols = [
    "avg_weekly_price", "real_price", "price_vs_median",
    "eurozone_hicp", "eurozone_unemployment_rate", "eurozone_cci",
    "hicp_lag1m", "unemployment_lag1m", "cci_lag1m",
    "product_age_weeks",
    "sales_lag1", "sales_lag2", "sales_lag4",
    "sales_rolling4_mean", "sales_rolling4_std",
]


STEP 8 — Standardise Continuous Features (fit on train only)


In [33]:
scaler = StandardScaler()
train[scale_cols] = scaler.fit_transform(train[scale_cols])
val[scale_cols]   = scaler.transform(val[scale_cols])
test[scale_cols]  = scaler.transform(test[scale_cols])
log(f"  Scaled {len(scale_cols)} continuous columns.")
log(f"  Scaler fitted on train only — no leakage into val/test.")

  Scaled 15 continuous columns.
  Scaler fitted on train only — no leakage into val/test.


In [34]:
log("\n" + "=" * 70)
log("STEP 9 — Save Outputs")
log("=" * 70)


STEP 9 — Save Outputs


In [35]:
df_full.to_csv("tshirts_preprocessed.csv", index=False)
log("  Saved: tshirts_preprocessed.csv  (full dataset, unscaled)")

  Saved: tshirts_preprocessed.csv  (full dataset, unscaled)


In [36]:
train.to_csv("tshirts_train.csv", index=False)
val.to_csv("tshirts_val.csv",     index=False)
test.to_csv("tshirts_test.csv",   index=False)
log("  Saved: tshirts_train.csv")
log("  Saved: tshirts_val.csv")
log("  Saved: tshirts_test.csv")

  Saved: tshirts_train.csv
  Saved: tshirts_val.csv
  Saved: tshirts_test.csv


In [37]:
with open(REPORT_PATH, "w") as f:
    f.write("\n".join(report_lines))
log(f"  Saved: {REPORT_PATH}")

  Saved: preprocessing_report.txt


In [38]:
log("\n" + "=" * 70)
log("FINAL SUMMARY")
log("=" * 70)
log(f"  Total rows (full grid): {len(df_full):,}")
log(f"  Total features:         {len(df_full.columns)}")
log(f"  Target columns:         weekly_sales_volume  (raw)  |  log_sales_volume  (for modeling)")
log(f"""
  FEATURE GROUPS:
    Macro (current):    eurozone_hicp, eurozone_unemployment_rate, eurozone_cci
    Macro (lagged 1m):  hicp_lag1m, unemployment_lag1m, cci_lag1m
    Temporal:           week_of_year, month, quarter, year,
                        is_spring_summer, is_sale_season
    Product:            product_age_weeks
    Price:              avg_weekly_price, real_price, price_vs_median
    Demand history:     sales_lag1, sales_lag2, sales_lag4,
                        sales_rolling4_mean, sales_rolling4_std
    Product attrs:      graphical_appearance_name_* (one-hot)
                        colour_group_name_* (one-hot)
                        perceived_colour_value_name_* (one-hot)
                        index_group_name_* (one-hot)
""")
log("  READY FOR MODELING.")


FINAL SUMMARY
  Total rows (full grid): 834,538
  Total features:         113
  Target columns:         weekly_sales_volume  (raw)  |  log_sales_volume  (for modeling)

  FEATURE GROUPS:
    Macro (current):    eurozone_hicp, eurozone_unemployment_rate, eurozone_cci
    Macro (lagged 1m):  hicp_lag1m, unemployment_lag1m, cci_lag1m
    Temporal:           week_of_year, month, quarter, year,
                        is_spring_summer, is_sale_season
    Product:            product_age_weeks
    Price:              avg_weekly_price, real_price, price_vs_median
    Demand history:     sales_lag1, sales_lag2, sales_lag4,
                        sales_rolling4_mean, sales_rolling4_std
    Product attrs:      graphical_appearance_name_* (one-hot)
                        colour_group_name_* (one-hot)
                        perceived_colour_value_name_* (one-hot)
                        index_group_name_* (one-hot)

  READY FOR MODELING.
